In [17]:
from pathlib import Path
import pandas as pd

# Завдання 1
incoming_dir = Path("data/incoming")
files = sorted(incoming_dir.glob("orders_*.jsonl"))

new_days = []

for path in files:
    new_days.append(pd.read_json(path, lines=True, dtype=False))


orders = pd.concat(new_days, ignore_index=True)
print(orders)
print(orders.shape)
orders.info()
print(orders["status"].value_counts(dropna = False))
print(orders["order_id"].nunique())

      order_id  customer_id country_code                 created_at  \
0       600001      10033.0           PL  2026-10-12T00:01:14+00:00   
1       600002      11084.0           UA  2026-10-12T00:04:42+00:00   
2       600003      12940.0           UA  2026-10-12T00:07:05+00:00   
3       600004      13720.0           PL  2026-10-12T00:14:11+00:00   
4       600005      13159.0           UA  2026-10-12T00:15:42+00:00   
...        ...          ...          ...                        ...   
2343    602295      10640.0           PL  2026-10-17T23:29:07+00:00   
2344    602296      12577.0           UA  2026-10-17T23:34:31+00:00   
2345    602297      10653.0           UA  2026-10-17T23:47:52+00:00   
2346    602298      13565.0           UA  2026-10-17T23:49:05+00:00   
2347    602299      12706.0           PL  2026-10-17T23:58:19+00:00   

                status  product_id  quantity  unit_price total_amount  
0              shipped         401         2     52999.0     105998.0  
1  

In [18]:
# Завдання 2

orders["total_amount"] = pd.to_numeric(orders["total_amount"].astype(str).str.replace(" ", "").str.replace(",", "."), errors="coerce")
orders["status"] = orders["status"].str.strip().str.lower().str.replace(" ", "_")
orders["created_at"] = pd.to_datetime(orders["created_at"], format="ISO8601", errors="coerce", utc=True)

orders = orders.dropna(subset=["total_amount", "created_at"])
orders = orders.drop_duplicates(subset="order_id", keep="last").reset_index(drop=True)

# report[["daily_sales ", "revenue"]].to_csv("with_index.csv", index=False)

In [19]:
# Завдання 3

products = pd.read_json("data/products.json")
products = products[["product_id", "category"]].drop_duplicates(subset=["product_id"])

orders = orders.merge(products, on="product_id", how="left")
orders["category"] = orders["category"].fillna("Невідомо")

In [20]:
# Завдання 4

daily_sales = orders[~orders["status"].isin(["cancelled", "returned_to_sender"])].copy()

daily_sales["order_date"] = daily_sales["created_at"].dt.date
daily_sales = daily_sales.groupby(["order_date", "category"]).agg(
    orders_cnt=("order_id", "count"),
    revenue=("total_amount", "sum")
).reset_index().sort_values(["order_date", "revenue"], ascending=[True, False])
print(daily_sales.head(10))

    order_date           category  orders_cnt    revenue
10  2026-10-12          Смартфони         114  4618877.0
6   2026-10-12           Ноутбуки          30  1565968.0
5   2026-10-12          Навушники          53   509743.0
9   2026-10-12  Розумні годинники          22   380478.0
8   2026-10-12           Планшети          11   319987.0
11  2026-10-12              Чохли          17    57475.0
4   2026-10-12               Миші           7    31493.0
0   2026-10-12   Зарядні пристрої          15    25179.0
7   2026-10-12         Павербанки          11    20986.0
3   2026-10-12         Клавіатури           4    19996.0


In [21]:
# Завдання 5

daily_sales[["order_date", "category", "orders_cnt", "revenue"]].to_csv("daily_sales.csv", index=False)
